<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Lasso Regression

*Session 5 · Notebook 02.06 · Lecture · Coach version*

## Overview

Ridge shrank coefficients toward zero but kept them all. **Lasso** goes further: it can drive some coefficients to **exactly zero**, dropping those features entirely. That makes Lasso a regulariser **and** an automatic feature selector, valuable when there are many candidate drivers and you want a smaller, more explainable model. This notebook shows the idea on a synthetic example where we know which features matter, contrasts Lasso with Ridge, applies it to California housing (with some deliberately useless candidate features added so we can watch Lasso weed them out), visualises the coefficient path, and tunes the penalty with `LassoCV`.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain the Lasso (L1) penalty and how it differs from Ridge (L2).
- See Lasso perform feature selection by zeroing irrelevant coefficients.
- Fit Lasso in a scaled workflow and compare it to Ridge and ordinary regression.
- Read a Lasso coefficient path (coefficients dropping to zero as alpha grows).
- Choose `alpha` with `LassoCV`, and weigh prediction accuracy against a parsimonious model.

## Prerequisites

- Session 5 notebook 02.05 (Ridge regression, the L2 penalty, coefficient paths, cross-validation).
- Comfort with `train_test_split` and standardisation.

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What is Lasso regression?](#sec2)
3. [Synthetic data: Lasso selects features](#sec3)
4. [Real data: California housing](#sec4)
5. [Lasso vs Ridge vs ordinary regression](#sec5)
6. [The effect of alpha (coefficient path)](#sec6)
7. [Choosing alpha with cross-validation](#sec7)
8. [Exercises](#exercises)
9. [Challenge](#challenge)
10. [Key Takeaways](#takeaways)
11. [Further Reading](#reading)

<a id="setup"></a>
# Section 0: Setup

We build a small synthetic example first, then use the **California housing** dataset (20,640 districts; predict the median house value from eight features). It is read from the repo-root `datasets/` folder (two levels up); a clean, modern alternative to the Boston set used in the Ridge notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Libraries ready.')

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Risk teams often start with a long list of candidate drivers and want to end with a short, defensible one.

| Lasso gives us... | Why a risk team cares |
|---|---|
| Automatic feature selection | A leaner scorecard using only the drivers that matter, easier to explain and govern |
| Sparse, readable models | Fewer coefficients to validate and monitor |
| Handling many candidate features | Sifts a wide feature set down to the useful few |
| Regularisation | Still curbs overfitting, like Ridge |

Where Ridge keeps every feature (just smaller), Lasso can say 'this driver adds nothing' and remove it, which is often exactly the decision a modeller needs help making.

<a id="sec2"></a>
# Section 2: What is Lasso regression?

**Definition:** Lasso regression is linear regression with an **L1 penalty**: it minimises the squared error **plus** `alpha` times the sum of the **absolute** coefficients. Because of the shape of the absolute-value penalty, Lasso can push coefficients all the way to **exactly zero**, removing those features.

**Example:** given ten candidate drivers where only three truly matter, Lasso keeps those three and sets the other seven to zero.

**Analogy:** Ridge is a leash that shortens every coefficient a little; Lasso is a **budget** that forces you to spend on only the most useful features and drop the rest entirely.

**Explanation:**

- **L1 vs L2:** Ridge (L2) penalises squared coefficients and shrinks smoothly; Lasso (L1) penalises absolute coefficients and can zero them. That zeroing is **feature selection**.
- **`alpha`** controls the strength: 0 is ordinary regression; larger `alpha` means more coefficients driven to zero (a sparser model).
- **Scaling is required** (as with Ridge), so features are penalised fairly.
- **Trade-off:** among strongly correlated features Lasso tends to keep one and drop the others, which is great for simplicity but can discard a useful driver; **ElasticNet** (L1+L2) is a middle ground.

**The formula (and why Lasso is the odd one out):**

Lasso minimises the squared error **plus** the L1 penalty (the sum of the *absolute* coefficients):

$$\text{minimise} \quad \sum_i (y_i - \hat{y}_i)^2 + \alpha \sum_j |\beta_j|$$

Unlike ordinary regression and Ridge, this has **no closed-form formula**. The absolute-value penalty has a sharp corner at zero and is not differentiable there, so you cannot just solve one equation for $\boldsymbol{\beta}$. Instead Lasso is solved by an **iterative optimiser** (coordinate descent) that sweeps through the coefficients over and over until they stop changing. That iteration is exactly why Lasso has a `max_iter` setting, and that non-differentiable corner is what lets coefficients land on **exactly zero** (feature selection). See the side-note notebook `05_18` for a from-scratch demonstration.

**scikit-learn documentation:** [`Lasso`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) and [`LassoCV`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LassoCV.html)

<a id="sec3"></a>
# Section 3: Synthetic data - Lasso selects features

We generate ten features where **only the first three** actually drive the target; the other seven are pure noise. We then fit Lasso and Ridge and compare what each does to the irrelevant coefficients. Lasso should zero them; Ridge should only shrink them.

In [ ]:
rng = np.random.default_rng(0)
n, p = 200, 10
X_syn = rng.normal(0, 1, size=(n, p))
true_coef = np.array([5.0, -3.0, 2.0, 0, 0, 0, 0, 0, 0, 0])   # only x0,x1,x2 matter
y_syn = X_syn @ true_coef + rng.normal(0, 1, size=n)

lasso_s = Lasso(alpha=0.1).fit(X_syn, y_syn)
ridge_s = Ridge(alpha=1.0).fit(X_syn, y_syn)

compare = pd.DataFrame({
    'feature': [f'x{i}' for i in range(p)],
    'true': true_coef,
    'lasso': lasso_s.coef_.round(2),
    'ridge': ridge_s.coef_.round(2),
})
print(compare.to_string(index=False))

print(f'\nLasso set {(lasso_s.coef_ == 0).sum()} coefficients to EXACTLY zero.')
print(f'Ridge set {(ridge_s.coef_ == 0).sum()} coefficients to exactly zero.')
print('Lasso recovered the three real drivers and dropped the seven noise features; '
      'Ridge kept all ten, just smaller.')

<a id="sec4"></a>
# Section 4: Real data - California housing

The target is `MedHouseVal`; the eight columns are the real features. To make the selection story concrete on real data, we also add **four candidate features that are pure noise**, the kind of useless columns a modeller might include 'just in case'. A good model should learn to ignore them. We split and standardise (fitting the scaler on the training data only).

In [ ]:
# Or read directly from the public S3 bucket (no local file needed):
# df = pd.read_csv('https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/california_housing.csv')
# ...or read the paths from a config file (the local read below stays the default):
# from config import session_datasets_http
# df = pd.read_csv(session_datasets_http["california_housing"])
# Or from S3 with Spark, then to pandas (needs a SparkSession, e.g. on Databricks):
# df = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/california_housing.csv", header=True, inferSchema=True).toPandas()
df = pd.read_csv('../../datasets/Session_5/california_housing.csv')
X = df.drop(columns='MedHouseVal').copy()
y = df['MedHouseVal']

# Add four useless candidate features to test whether Lasso weeds them out
noise_rng = np.random.default_rng(7)
noise_cols = [f'noise_{i}' for i in range(1, 5)]
for col in noise_cols:
    X[col] = noise_rng.normal(0, 1, len(X))

print('Real features + 4 noise candidates:', list(X.columns))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

<a id="sec5"></a>
# Section 5: Lasso vs Ridge vs ordinary regression

We fit all three on the same scaled data and compare test performance and, crucially, how many coefficients each keeps. Ordinary regression and Ridge keep all twelve (including the four noise features); Lasso removes the useless ones.

In [ ]:
lr = LinearRegression().fit(X_train_scaled, y_train)
ridge = Ridge(alpha=1.0).fit(X_train_scaled, y_train)
lasso = Lasso(alpha=0.02, max_iter=10000).fit(X_train_scaled, y_train)

for name, model in [('Ordinary', lr), ('Ridge (a=1)', ridge), ('Lasso (a=0.02)', lasso)]:
    pred = model.predict(X_test_scaled)
    nonzero = int((model.coef_ != 0).sum())
    print(f'{name:15} test R2 = {r2_score(y_test, pred):.3f}, '
          f'RMSE = {np.sqrt(mean_squared_error(y_test, pred)):.3f}, '
          f'features kept = {nonzero}/{len(model.coef_)}')

In [ ]:
# Did Lasso drop the four noise features we planted?
lasso_coef = pd.Series(lasso.coef_, index=X.columns)
dropped = lasso_coef[lasso_coef == 0].index.tolist()
print('Lasso dropped:', dropped)
print('\nAll four noise features were removed, at the same test R2 as the full model: a '
      'simpler, more honest model at no accuracy cost.')

<a id="sec6"></a>
# Section 6: The effect of alpha (coefficient path)

As with Ridge, we plot each coefficient against `alpha`. The difference is the shape: Lasso coefficients drop to **exactly zero** one after another as `alpha` grows (Ridge's shrank smoothly but never hit zero). The noise features collapse to zero almost immediately; the real drivers persist to much larger `alpha`.

In [ ]:
alphas = np.logspace(-3, 1, 60)
paths = np.array([Lasso(alpha=a, max_iter=10000).fit(X_train_scaled, y_train).coef_
                  for a in alphas])

plt.figure(figsize=(9, 5))
for i, col in enumerate(X.columns):
    style = ':' if col.startswith('noise') else '-'
    plt.plot(alphas, paths[:, i], style, label=col)
plt.xscale('log'); plt.xlabel('alpha (log scale)'); plt.ylabel('coefficient')
plt.axhline(0, color='black', lw=0.8)
plt.title('Lasso coefficient paths (noise features dotted, and first to hit 0)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

<a id="sec7"></a>
# Section 7: Choosing alpha with cross-validation

`LassoCV` tries many `alpha` values with cross-validation and keeps the one with the best predictive score. A subtlety worth knowing: because it optimises **prediction**, it often settles on a small `alpha` that keeps near-useless features with tiny coefficients (dropping them barely changes the score). If you want a **parsimonious, explainable** model, a slightly stronger `alpha` (like the 0.02 above) removes the noise at almost no accuracy cost. (Cross-validation itself is introduced in notebook 11.)

In [ ]:
lasso_cv = LassoCV(alphas=np.logspace(-3, 1, 60), cv=5, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

pred = lasso_cv.predict(X_test_scaled)
cv_dropped = [c for c in noise_cols if pd.Series(lasso_cv.coef_, index=X.columns)[c] == 0]
print('Best alpha by cross-validation:', round(lasso_cv.alpha_, 4))
print(f'Tuned Lasso test R2 = {r2_score(y_test, pred):.3f}, features kept = '
      f'{(lasso_cv.coef_ != 0).sum()}/{len(lasso_cv.coef_)}')
print('Noise features it dropped at this alpha:', cv_dropped)
print('\nThe prediction-optimal alpha is small and leaves most noise in with tiny weights. '
      'For a sparse, auditable model, prefer a slightly stronger alpha as in Section 5.')

<a id="exercises"></a>
# Section 8: Exercises

### Exercise 1: A stronger penalty drops more features

Fit a Lasso with `alpha=0.1` (use `max_iter=10000`) on the scaled training data and print how many features it keeps. Compare with the `alpha=0.02` model from Section 5.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
strong = Lasso(alpha=0.1, max_iter=10000).fit(X_train_scaled, y_train)
print('alpha=0.1  features kept:', int((strong.coef_ != 0).sum()))
print('alpha=0.02 features kept:', int((lasso.coef_ != 0).sum()))
print('A larger alpha drives more coefficients to zero (a sparser model).')

### Exercise 2: Ridge keeps everything

Fit a Ridge with `alpha=1.0` on the scaled training data and count its non-zero coefficients. Confirm Ridge does not perform feature selection (it keeps the noise features too).

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
r = Ridge(alpha=1.0).fit(X_train_scaled, y_train)
print('Ridge features kept:', int((r.coef_ != 0).sum()), 'of', len(r.coef_))
print('Ridge shrinks but never zeroes, so it keeps every feature, noise included.')

### Exercise 3: The selected drivers

From the `alpha=0.02` Lasso model, print the features it kept (non-zero coefficient) sorted by absolute coefficient size. Are the noise features absent?

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
kept = pd.Series(lasso.coef_, index=X.columns)
kept = kept[kept != 0].sort_values(key=abs, ascending=False)
print(kept.round(3))
print('The noise_* features do not appear: Lasso removed them.')

<a id="challenge"></a>
## Challenge (optional): sparsity vs accuracy, Lasso vs Ridge

Does the simpler Lasso model cost accuracy? Compare tuned Ridge (`RidgeCV`, keeps all features) against the sparse `alpha=0.02` Lasso, using 5-fold cross-validation on the whole dataset (put scaling inside a pipeline to stay leakage-free). Report each model's mean cross-validated R-squared and how many features the Lasso keeps, and comment on the trade-off.

In [ ]:
# Your turn. Write your solution here:


In [ ]:
# Coach answer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

ridge_pipe = Pipeline([('scaler', StandardScaler()),
                       ('model', RidgeCV(alphas=np.logspace(-3, 3, 50)))])
lasso_pipe = Pipeline([('scaler', StandardScaler()),
                       ('model', Lasso(alpha=0.02, max_iter=10000))])

# cross_val_score clones each pipeline, fits the clones on folds, and discards them:
# it returns only the scores and leaves ridge_pipe / lasso_pipe themselves UNFITTED.
ridge_r2 = cross_val_score(ridge_pipe, X, y, cv=5, scoring='r2').mean()
lasso_r2 = cross_val_score(lasso_pipe, X, y, cv=5, scoring='r2').mean()

# Refit Lasso on all the data purely so we can inspect its coefficients and count how many
# features it kept. (Ridge keeps every feature by definition, so we report X.shape[1] for it
# and never need to refit or inspect its coefficients.)
lasso_pipe.fit(X, y)
kept = int((lasso_pipe.named_steps['model'].coef_ != 0).sum())

print(f'Ridge (all features)  CV R2: {ridge_r2:.3f}  (keeps {X.shape[1]} features)')
print(f'Lasso (alpha=0.02)    CV R2: {lasso_r2:.3f}  (keeps {kept}/{X.shape[1]} features)')
print('\nThe Lasso reaches essentially the same accuracy while using fewer features (it drops '
      'the noise). When some candidate features are irrelevant, that is a clear win for Lasso: '
      'a simpler, more auditable model at no real cost.')

<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| Lasso regression | Linear regression + an L1 penalty on absolute coefficient size |
| L1 vs L2 | Lasso can zero coefficients (feature selection); Ridge only shrinks |
| `alpha` | Penalty strength: larger means more coefficients set to zero (sparser) |
| `Lasso(alpha=..., max_iter=...)` | Fit a Lasso model |
| `(coef_ != 0).sum()` | Count the features the model kept |
| coefficient path | Coefficients drop to exactly 0 one by one as alpha grows |
| `LassoCV` | Choose alpha by cross-validation (optimises prediction) |
| prediction vs parsimony | The predictive-best alpha may keep noise; a stronger alpha gives a sparser model |
| ElasticNet | L1 + L2 combined, a middle ground |


## Conclusion

You can now use Lasso to regularise **and** select features, contrast its L1 penalty with Ridge's L2, read a coefficient path where features drop out, tune the penalty with cross-validation, and reason about prediction versus a parsimonious model. Together with Ridge, this completes the regularisation toolkit; the regression capstone practical then brings the whole workflow together.

<a id="reading"></a>
## Further Reading & Resources

- [scikit-learn: Lasso](https://scikit-learn.org/stable/modules/linear_model.html#lasso) the L1 penalty and LassoCV.
- [scikit-learn: Lasso path example](https://scikit-learn.org/stable/auto_examples/linear_model/plot_lasso_coordinate_descent_path.html) the coefficient path.
- [scikit-learn: ElasticNet](https://scikit-learn.org/stable/modules/linear_model.html#elastic-net) combining L1 and L2.